### Surface Diffusion Barrier

The nudged elastic band (NEB) method finds minimum energy pathways between two states.
Here we calculate the diffusion barrier for an oxygen atom moving between fcc and hcp sites on Pt(111).

In [ ]:
from vasp import Vasp
from vasp.runners import LocalRunner
from ase.build import fcc111, add_adsorbate
from ase.constraints import FixAtoms
from ase.neb import NEB
import matplotlib.pyplot as plt

# Create and relax initial state: O at fcc site
initial = fcc111('Pt', size=(2, 2, 3), vacuum=10.0)
add_adsorbate(initial, 'O', height=1.2, position='fcc')
constraint = FixAtoms(mask=[atom.symbol != 'O' for atom in initial])
initial.set_constraint(constraint)

calc_init = Vasp('surfaces/neb-init',
                 xc='PBE', encut=300, kpts=[3, 3, 1],
                 ibrion=2, nsw=15, isif=0,
                 atoms=initial)
e_init = calc_init.get_potential_energy()
initial = calc_init.atoms.copy()
print(f'Initial (fcc): {e_init:.4f} eV')
print(f'Initial O position: {initial.positions[-1]}')

# Create and relax final state: O at hcp site
final = fcc111('Pt', size=(2, 2, 3), vacuum=10.0)
add_adsorbate(final, 'O', height=1.2, position='hcp')
constraint_final = FixAtoms(mask=[atom.symbol != 'O' for atom in final])
final.set_constraint(constraint_final)

calc_final = Vasp('surfaces/neb-final',
                  xc='PBE', encut=300, kpts=[3, 3, 1],
                  ibrion=2, nsw=15, isif=0,
                  atoms=final)
e_final = calc_final.get_potential_energy()
final = calc_final.atoms.copy()
print(f'Final (hcp): {e_final:.4f} eV')
print(f'Final O position: {final.positions[-1]}')

In [ ]:
# Create NEB band with 3 intermediate images
# Important: use copies of the relaxed structures
n_intermediate = 3
images = [initial.copy()]
for i in range(n_intermediate):
    images.append(initial.copy())
images.append(final.copy())

# Interpolate positions between initial and final
neb = NEB(images)
neb.interpolate()

print(f'Total images: {len(images)} (including endpoints)')
print(f'Intermediate images: {n_intermediate}')
print('\nInterpolated O positions:')
for i, img in enumerate(images):
    print(f'  Image {i}: {img.positions[-1]}')

In [ ]:
# Run NEB calculation
# Use 3 MPI processes (one per intermediate image) for proper NEB parallelization
# The --oversubscribe flag allows running on a single CPU machine
neb_runner = LocalRunner(
    nprocs=3, 
    mpi_extra_args='--oversubscribe --mca mpi_cuda_support 0 --mca btl_base_warn_component_unused 0'
)

calc_neb = Vasp('surfaces/Pt-O-neb',
                xc='PBE', encut=300, kpts=[3, 3, 1],
                ibrion=1, nsw=40,
                spring=-5.0,
                runner=neb_runner,
                atoms=images)

# Trigger the calculation first
calc_neb.calculate()

# Now read NEB results
neb_images, energies = calc_neb.get_neb()

print('NEB energies (eV):')
for i, e in enumerate(energies):
    if e is not None:
        print(f'  Image {i}: {e:.4f}')

# Calculate barrier
valid_energies = [e for e in energies if e is not None]
if valid_energies:
    barrier = max(valid_energies) - valid_energies[0]
    print(f'\nDiffusion barrier: {barrier:.3f} eV')

In [ ]:
# Plot NEB profile
calc_neb.plot_neb(show=True)